# GTEx V11: cross-region gene-expression prediction in unseen donors
**BMEN 600 | Public-reviewer Google Colab notebook**

**Research question:** How accurately can gene expression in one human brain region predict expression in another in unseen GTEx donors, and how does predictive performance vary across region pairs?

**Dataset:** GTEx V11 gene-read-count files for 12 human brain regions. The archive also contains a cervical spinal cord file, which is **excluded** from the brain-region analysis. No other dataset or patient cohort is used.

**Reproduce:** Open in [Google Colab](https://colab.research.google.com/) and select **Runtime → Run all**. Section 2 downloads `gene_reads_GTEx.zip` from the GitHub release `ellamichele/BMEN600Group23`, tag `v1.0-data`, and extracts 13 `.gct.gz` files without a manual upload. The release asset must be publicly accessible for anonymous reviewers.

**Outputs:** Dataset sample/donor counts, donor overlap, region-pair prediction metrics on **held-out donors**, a mean-expression baseline comparison, per-gene held-out R², plots, a compact alternative to Table 2, a new combined Figure 8 (panels A–B), and a reproducibility metadata file. The default run evaluates six prespecified prediction directions; set `RUN_ALL_PAIRS = True` to attempt all eligible directions (substantially longer runtime).

**Provenance and scope:** Repository maintainers should record the original GTEx V11 source, acquisition date and dataset-sharing terms. Results measure predictive performance under the specified pipeline; they are not an intrinsic measure of brain-region similarity. Different region pairs may involve different donors and selected target genes; pairwise comparisons are therefore exploratory until tested using matched cohorts and common targets.

**Analysis scope and interpretation:** Donors—not samples or genes—are independent cases. The primary outcomes are median gene-wise test R² and prediction skill relative to a training-data mean baseline, supplemented by MSE, RMSE and MAE. A held-out donor is never used to select genes, tune the model, or fit scaling parameters. Bootstrap intervals resample held-out donors conditional on a fitted model. The notebook compares predictive performance between GTEx brain-region directions; no additional biological-characteristic association analysis is performed.

**Results notice:** All result tables and figures are generated when the notebook is run. The downloaded results archive excludes files explicitly marked `PRIVATE` that contain donor identifiers.

## 1. Scientific rationale and evaluation design
GTEx provides gene-expression measurements from multiple human tissues, including matched brain regions sampled from the same donors [1]. We test whether expression in a source brain region predicts expression in a target brain region for **donors not seen during model development**. We examine both the prediction accuracy for each direction and variation in accuracy across directions.

The **donor** is the unit of independence. Expression matrices are aligned using IDs shared by the source and target brain regions. The outer split withholds 20% of paired donors; an inner split within development donors selects the Ridge penalty, while training-only gene selection and scaling avoid leakage. Ridge regression regularizes high-dimensional, correlated predictors [3], and separate validation and testing reduces selection bias [4].

We normalize raw gene counts within each sample using log₂(CPM + 1). This is not TPM or TMM; library-size scaling may retain compositional bias [2]. Available count files alone do not provide all donor and sample-level covariates (e.g., age, sex, tissue quality, batch). These factors may contribute to apparent prediction accuracy. Since eligible donors and selected target genes differ across pairs, performance comparisons should be described as exploratory rather than definitive rankings. Repeated donor splits and common-donor/common-gene sensitivity analyses would strengthen conclusions.

In [ ]:
from pathlib import Path
import gzip, json, hashlib, platform, itertools, shutil
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import sklearn
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from IPython.display import display

SEED = 42
RUN_ALL_PAIRS = False
INCLUDE_SPINAL_CORD = False
SOURCE = 'caudate_basal_ganglia'
TARGET = 'putamen_basal_ganglia'
MIN_PAIRED_DONORS = 60
N_SOURCE_GENES = 500
N_TARGET_GENES = 1000
ALPHAS = [1., 10., 100., 1000.]
N_BOOTSTRAPS = 1000

OUT = Path('/content/gtex_results')
OUT.mkdir(parents=True, exist_ok=True)
# Delete previous outputs so a rerun never packages stale files from an older analysis.
for old_file in OUT.iterdir():
    if old_file.is_file():
        old_file.unlink()

plt.rcParams.update({'figure.dpi': 120, 'axes.spines.top': False, 'axes.spines.right': False})

# Illustrative directional region pairs for the quicker default reviewer run.
ILLUSTRATIVE_PAIRS = [
    ('caudate_basal_ganglia', 'putamen_basal_ganglia'),
    ('putamen_basal_ganglia', 'caudate_basal_ganglia'),
    ('cortex', 'frontal_cortex_ba9'),
    ('frontal_cortex_ba9', 'cortex'),
    ('cerebellum', 'cerebellar_hemisphere'),
    ('cerebellar_hemisphere', 'cerebellum')
]

## 2. Download the GTEx V11 count files automatically
The cell below downloads the release asset **`gene_reads_GTEx.zip`** from the public repository **`ellamichele/BMEN600Group23`**, release tag **`v1.0-data`**. It extracts 13 `.gct.gz` files to `/content/gtex_input`, validates their filenames and file headers, and prepares `paths` for the loading code. The ZIP may contain folders, but it must contain the original GTEx `.gct.gz` filenames. For anonymous reviewer access, the release and asset must be public.

In [ ]:
from pathlib import Path
import requests
import zipfile
import shutil

# Public GitHub Release containing the GTEx input data.
GITHUB_OWNER = "ellamichele"
GITHUB_REPO = "BMEN600Group23"
RELEASE_TAG = "v1.0-data"
ZIP_NAME = "gene_reads_GTEx.zip"

DATA_URL = (
    f"https://github.com/{GITHUB_OWNER}/{GITHUB_REPO}/"
    f"releases/download/{RELEASE_TAG}/{ZIP_NAME}"
)
DATA_DIR = Path("/content/gtex_input")
DATA_DIR.mkdir(parents=True, exist_ok=True)
ZIP_PATH = Path("/content") / ZIP_NAME

print(f"Downloading GTEx V11 data from: {DATA_URL}")
try:
    with requests.get(DATA_URL, stream=True, timeout=(30, 180),
                      headers={"User-Agent": "GTEx-BMEN600-reproducibility-notebook"}) as response:
        response.raise_for_status()
        with ZIP_PATH.open("wb") as output:
            for chunk in response.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    output.write(chunk)
except requests.RequestException as exc:
    raise RuntimeError(
        "GitHub dataset download failed. Confirm that the repository and Release "
        "are public and that the Release contains the asset gene_reads_GTEx.zip. "
        f"Attempted URL: {DATA_URL}"
    ) from exc

# Extract expected data files only. Flatten optional directories inside ZIP.
# Do not extract arbitrary paths, and reject duplicate region filenames.
try:
    with zipfile.ZipFile(ZIP_PATH) as archive:
        members = [m for m in archive.infolist()
                   if not m.is_dir() and Path(m.filename).name.endswith(".gct.gz")]
        names = [Path(m.filename).name for m in members]
        if len(names) != len(set(names)):
            raise ValueError("ZIP contains duplicate .gct.gz filenames.")
        if len(names) != 13:
            raise ValueError(f"Expected 13 .gct.gz files in ZIP, found {len(names)}")
        unexpected = [name for name in names if not
                      (name.startswith("gene_reads_adult_gtex_v11_brain_") and
                       name.endswith(".gct.gz"))]
        if unexpected:
            raise ValueError(
                "Unexpected dataset filename(s): " + ", ".join(unexpected)
                + ". Ensure the files retain their original names (no '(1)' suffixes)."
            )
        for m in members:
            destination = DATA_DIR / Path(m.filename).name
            with archive.open(m) as source, destination.open("wb") as target:
                shutil.copyfileobj(source, target)
except (zipfile.BadZipFile, OSError, ValueError) as exc:
    raise RuntimeError("Unable to validate/extract GTEx release ZIP: " + str(exc)) from exc

paths = sorted(DATA_DIR.glob("gene_reads_adult_gtex_v11_brain_*.gct.gz"))
assert len(paths) == 13, f"Expected 13 region files after extraction, found {len(paths)}"
for path in paths:
    with path.open("rb") as f:
        if f.read(2) != b"\x1f\x8b":
            raise ValueError(f"Not a valid gzip file header: {path.name}")
print(f"Dataset ready: {len(paths)} region files downloaded and identified.")
print("Run Section 3 to load, validate, and normalize the count matrices.")

## 3. Load, validate and normalize
A GCT 1.2 file has a version line, a dimensions line, then the table. `Name` identifies genes and `Description` supplies symbols. GTEx donor IDs are the first two hyphen-separated fields of the sample ID. Multiple samples for one donor within a region are averaged **after sample-level normalization**; the summary records this choice. We retain full Ensembl IDs to avoid collisions.

In [ ]:
def load_region(path):
    with gzip.open(path, 'rt') as f:
        version = f.readline().strip()
        n_genes, n_samples = map(int, f.readline().split())
    if version != '#1.2':
        raise ValueError(f'Unsupported GCT version: {version}')
    table = pd.read_csv(path, sep='\t', skiprows=2)
    if table.shape != (n_genes, n_samples + 2):
        raise ValueError(f'Dimension mismatch: {path.name}')
    if table['Name'].duplicated().any():
        raise ValueError('Duplicate gene IDs')
    symbols = table.set_index('Name')['Description']
    counts = table.iloc[:, 2:].astype('float32')
    counts.index = table['Name']
    if not np.isfinite(counts.to_numpy()).all() or (counts.to_numpy() < 0).any():
        raise ValueError('Counts must be finite and nonnegative')
    libraries = counts.sum(axis=0)
    if (libraries <= 0).any():
        raise ValueError('Zero-count sample found')
    donors = []
    for sample in counts.columns:
        parts = sample.split('-')
        if len(parts) < 3 or parts[0] != 'GTEX':
            raise ValueError(f'Unexpected sample ID: {sample}')
        donors.append('-'.join(parts[:2]))
    logcpm = np.log2(counts.div(libraries, axis=1) * 1e6 + 1).T
    logcpm.index = donors
    matrix = logcpm.groupby(level=0, sort=True).mean().astype('float32')
    record = dict(region=path.name.removeprefix('gene_reads_adult_gtex_v11_brain_').removesuffix('.gct.gz'),
                  genes=n_genes, samples=n_samples, donors=len(matrix),
                  extra_samples=n_samples-len(matrix), median_library_reads=float(libraries.median()),
                  sha256=hashlib.sha256(path.read_bytes()).hexdigest())
    return matrix, symbols, record

regions, symbols, records = {}, {}, []
for path in paths:
    matrix, gene_symbols, record = load_region(path)
    regions[record['region']] = matrix
    symbols[record['region']] = gene_symbols
    records.append(record)
    print(record['region'], matrix.shape)
summary = pd.DataFrame(records)
summary.to_csv(OUT / 'table_1_dataset_summary.csv', index=False)
display(summary.drop(columns='sha256'))
active = sorted(r for r in regions if INCLUDE_SPINAL_CORD or not r.startswith('spinal_cord'))
overlap = pd.DataFrame([[len(regions[a].index.intersection(regions[b].index)) for b in active] for a in active], index=active, columns=active)
overlap.to_csv(OUT / 'donor_overlap.csv')

## 4. Dataset graphs
**Figure 1.** Unique donors per uploaded region after within-region aggregation. Counts describe available data, not the number used in every pair.

**Figure 2.** Donors with measurements in both regions. Each pair can have a different cohort. This limits direct comparison between pair scores.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 6))
s = summary.sort_values('donors')
ax.barh(s.region.str.replace('_', ' '), s.donors, color='#357da8')
ax.set(xlabel='Unique donors', title='Figure 1. Available donors by region')
fig.tight_layout(); fig.savefig(OUT / 'figure_1_region_counts.png', dpi=200); plt.show()
fig, ax = plt.subplots(figsize=(11, 9))
im = ax.imshow(overlap, cmap='Blues')
labels = [r.replace('_', ' ') for r in active]
ax.set_xticks(range(len(active)), labels, rotation=90)
ax.set_yticks(range(len(active)), labels)
for i in range(len(active)):
    for j in range(len(active)):
        ax.text(j, i, str(overlap.iloc[i,j]), ha='center', va='center', fontsize=7)
ax.set_title('Figure 2. Paired donor availability')
fig.colorbar(im, ax=ax, label='Paired donors')
fig.tight_layout(); fig.savefig(OUT / 'figure_2_donor_overlap.png', dpi=200); plt.show()

## 5. Cross-region prediction and evaluation
For each source → target direction, restrict the data to donors with measurements in **both** regions. Hold out 20% of those donors for final testing, then use an inner validation split to select the Ridge `alpha`. Select source and target genes independently **using the inner training donors only**: retain genes with expression equivalent to CPM > 1 in at least 20% of training donors, then select the highest-variance genes. Fit scaling parameters on training data only; once `alpha` is chosen, refit scaling and Ridge on the complete development set.

**Evaluation metrics:** A baseline predicts the development-set mean for each selected target gene. Baseline-relative prediction skill is `1 − model MSE / baseline MSE` averaged across donors and target genes (positive = better than baseline). Gene-wise held-out **R²** uses the test-donor mean of each gene as its reference; the median across selected genes summarizes the pair. MSE, RMSE and MAE are also reported for the selected targets. These measurements quantify predictive performance, not a causal relationship between regions.

Bootstrap intervals repeatedly resample **held-out donors**, holding model predictions fixed; they describe test-cohort uncertainty conditional on the fitted model, not variation from retraining. Genes are not independent donors. Pair comparisons can be affected by differing donor groups and selected target genes: treat rankings as exploratory and use repeated splits/common-target comparisons for stronger conclusions.

In [ ]:
def choose_genes(frame, n):
    # Filter out rarely expressed genes, then select high-variance predictors or targets.
    expressed = (frame > 1).mean(axis=0) >= 0.2  # equivalent to CPM > 1
    variance = frame.var(axis=0)
    ranked = variance[expressed & (variance > 1e-8)].sort_values(ascending=False)
    if len(ranked) < 10:
        raise ValueError('Too few expressed variable genes for modelling')
    return ranked.index[:n]


def analyze_pair(source, target):
    # All rows correspond to paired donor IDs in the same order.
    ids = regions[source].index.intersection(regions[target].index).sort_values()
    if len(ids) < MIN_PAIRED_DONORS:
        return None

    dev, test = train_test_split(np.arange(len(ids)), test_size=0.2, random_state=SEED)
    inner, val = train_test_split(dev, test_size=0.25, random_state=SEED)
    Xframe = regions[source].loc[ids]
    Yframe = regions[target].loc[ids]

    # Feature/target selection never uses validation or test donors.
    xgenes = choose_genes(Xframe.iloc[inner], N_SOURCE_GENES)
    ygenes = choose_genes(Yframe.iloc[inner], N_TARGET_GENES)
    X = Xframe[xgenes].to_numpy(dtype='float64')
    Y = Yframe[ygenes].to_numpy(dtype='float64')

    scaler = StandardScaler().fit(X[inner])
    Xi, Xv = scaler.transform(X[inner]), scaler.transform(X[val])
    validation = []
    for alpha in ALPHAS:
        fit = Ridge(alpha=alpha, solver='cholesky').fit(Xi, Y[inner])
        validation.append(np.mean((Y[val] - fit.predict(Xv))**2))
    best = ALPHAS[int(np.argmin(validation))]

    # Final model: refit on development donors only.
    scaler = StandardScaler().fit(X[dev])
    model = Ridge(alpha=best, solver='cholesky').fit(scaler.transform(X[dev]), Y[dev])
    truth = Y[test]
    pred = model.predict(scaler.transform(X[test]))
    baseline = np.broadcast_to(Y[dev].mean(axis=0), truth.shape)

    model_loss = ((truth - pred)**2).mean(axis=1)
    base_loss = ((truth - baseline)**2).mean(axis=1)
    skill = 1 - model_loss.mean() / base_loss.mean()
    model_mae = float(np.mean(np.abs(truth - pred)))
    model_rmse = float(np.sqrt(model_loss.mean()))

    # Donor bootstrap for baseline-relative skill.
    rng = np.random.default_rng(SEED)
    sampled = rng.integers(0, len(test), size=(N_BOOTSTRAPS, len(test)))
    bs = 1 - model_loss[sampled].mean(axis=1) / base_loss[sampled].mean(axis=1)
    lo, hi = np.quantile(bs, [0.025, 0.975])

    # R² is calculated independently for each selected target gene.
    sse = ((truth - pred)**2).sum(axis=0)
    sst = ((truth - truth.mean(axis=0))**2).sum(axis=0)
    per_gene_r2 = 1 - np.divide(
        sse, sst, out=np.full_like(sse, np.nan), where=sst > 1e-12
    )
    gene_table = pd.DataFrame({
        'source': source,
        'target': target,
        'gene_id': ygenes,
        'symbol': symbols[target].reindex(ygenes).values,
        'test_r2': per_gene_r2,
        'test_mae': np.abs(truth - pred).mean(axis=0),
        'test_rmse': np.sqrt(((truth - pred)**2).mean(axis=0)),
        'baseline_mse': ((truth - baseline)**2).mean(axis=0)
    })

    # Conditional bootstrap interval for the median of per-gene test R².
    r2_boot = []
    for sample in sampled:
        t, pr = truth[sample], pred[sample]
        den = ((t - t.mean(axis=0))**2).sum(axis=0)
        rr = 1 - np.divide(
            ((t - pr)**2).sum(axis=0), den,
            out=np.full_like(den, np.nan), where=den > 1e-12
        )
        r2_boot.append(np.nanmedian(rr))
    r2lo, r2hi = np.nanquantile(r2_boot, [0.025, 0.975])

    prefix = source + '__to__' + target
    gene_table.to_csv(OUT / (prefix + '_genes.csv'), index=False)
    # Retain donor-level records locally; exclude files ending in PRIVATE from public ZIP.
    pd.DataFrame({
        'donor_id': ids[test], 'model_mse': model_loss, 'baseline_mse': base_loss
    }).to_csv(OUT / (prefix + '_test_errors_PRIVATE.csv'), index=False)
    pd.DataFrame({
        'donor_id': ids,
        'split': ['test' if i in test else 'development' for i in range(len(ids))]
    }).to_csv(OUT / (prefix + '_splits_PRIVATE.csv'), index=False)

    row = dict(
        source=source, target=target,
        n_paired=len(ids), n_development=len(dev), n_test=len(test),
        n_source_genes=len(xgenes), n_target_genes=len(ygenes), alpha=best,
        model_mse=model_loss.mean(), model_rmse=model_rmse, model_mae=model_mae,
        baseline_mse=base_loss.mean(), skill=skill, ci_low=lo, ci_high=hi,
        median_gene_r2=float(np.nanmedian(per_gene_r2)),
        r2_ci_low=float(r2lo), r2_ci_high=float(r2hi)
    )
    return row, dict(truth=truth, pred=pred, genes=gene_table)


pairs = (list(itertools.permutations(active, 2)) if RUN_ALL_PAIRS
         else [(a, b) for a, b in ILLUSTRATIVE_PAIRS if a in active and b in active])
if not pairs and SOURCE in active and TARGET in active:
    pairs = [(SOURCE, TARGET)]

results, skipped, example, gene_results = [], [], None, []
for source, target in pairs:
    if source not in active or target not in active:
        raise ValueError('Selected region unavailable or excluded')
    output = analyze_pair(source, target)
    if output is None:
        skipped.append({'source': source, 'target': target, 'reason': 'too few paired donors'})
        continue
    row, detail = output
    results.append(row)
    gene_results.append(detail['genes'])
    if example is None:
        example = (row, detail)
    print(source, '→', target,
          'median gene R²:', round(row['median_gene_r2'], 3),
          '| skill:', round(row['skill'], 3))

if not results:
    raise ValueError('No eligible pairs. Inspect overlap and minimum donor threshold.')

results = pd.DataFrame(results)
results.to_csv(OUT / 'table_2_prediction_results.csv', index=False)
pd.DataFrame(skipped, columns=['source', 'target', 'reason']).to_csv(
    OUT / 'skipped_pairs.csv', index=False
)
display(results)
all_genes = pd.concat(gene_results, ignore_index=True)
all_genes.to_csv(OUT / 'table_3_gene_prediction_results.csv', index=False)

## 6. Results: compare prediction accuracy across region pairs
**Figure 3:** Baseline-relative prediction skill with conditional 95% donor-bootstrap intervals. The vertical zero line is the mean-expression baseline; a negative score means the model performs worse than the baseline.

**Figure 4:** Measured versus predicted expression for one illustrative target gene in held-out donors. The dashed line is perfect prediction.

**Figure 5:** Median gene-wise held-out R² with conditional 95% donor-bootstrap intervals for each tested direction.

**Figure 6 (optional all-pair mode):** Heatmap of directional prediction skill, shown when `RUN_ALL_PAIRS = True`. Rows are source regions and columns are target regions. These comparisons can differ in donors and selected genes.

In [ ]:
# Figure 3: baseline-relative prediction skill for tested pairs.
r = results.sort_values('skill')
fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(r))))
y = np.arange(len(r))
ax.hlines(y, r.ci_low, r.ci_high, color='#357da8')
ax.scatter(r.skill, y, color='#163b54')
ax.set_yticks(y, [a.replace('_', ' ') + ' → ' + b.replace('_', ' ')
                  for a, b in zip(r.source, r.target)])
ax.axvline(0, color='gray', linestyle='--')
ax.set(xlabel='Skill relative to development-set target mean',
       title='Figure 3. Held-out donor prediction skill (95% intervals)')
fig.tight_layout()
fig.savefig(OUT / 'figure_3_prediction_skill.png', dpi=200)
plt.show()

# Figure 4: predicted versus observed expression for one target gene.
row, detail = example
j = 0  # First target gene: ranked highest in inner-training-set variance.
truth, pred = detail['truth'][:, j], detail['pred'][:, j]
fig, ax = plt.subplots(figsize=(5, 5))
ax.scatter(truth, pred, alpha=0.8, color='#357da8')
limits = [min(truth.min(), pred.min()), max(truth.max(), pred.max())]
ax.plot(limits, limits, '--', color='gray')
name = detail['genes'].iloc[j]['symbol']
ax.set(xlabel='Observed log₂(CPM + 1)', ylabel='Predicted log₂(CPM + 1)',
       title=f'Figure 4. Held-out donors: {name}')
fig.tight_layout()
fig.savefig(OUT / 'figure_4_example_gene.png', dpi=200)
plt.show()

# Figure 5: median gene-wise held-out R² for tested pairs.
r = results.sort_values('median_gene_r2')
fig, ax = plt.subplots(figsize=(10, max(3, 0.35 * len(r))))
y = np.arange(len(r))
ax.hlines(y, r.r2_ci_low, r.r2_ci_high, color='#357da8')
ax.scatter(r.median_gene_r2, y, color='#163b54')
ax.set_yticks(y, [a.replace('_', ' ') + ' → ' + b.replace('_', ' ')
                  for a, b in zip(r.source, r.target)])
ax.axvline(0, color='gray', linestyle='--')
ax.set(xlabel='Median gene-wise test R²',
       title='Figure 5. Cross-region prediction R² (95% intervals)')
fig.tight_layout()
fig.savefig(OUT / 'figure_5_gene_r2.png', dpi=200)
plt.show()

# Figure 6: optional all-pair matrix of baseline-relative skill.
if RUN_ALL_PAIRS:
    heat = results.pivot(index='source', columns='target', values='skill').reindex(
        index=active, columns=active
    )
    fig, ax = plt.subplots(figsize=(11, 9))
    bound = max(1, float(np.nanmax(np.abs(heat.to_numpy()))))
    im = ax.imshow(np.ma.masked_invalid(heat.to_numpy()),
                   cmap='RdBu', vmin=-bound, vmax=bound)
    labels = [r.replace('_', ' ') for r in active]
    ax.set_xticks(range(len(active)), labels, rotation=90)
    ax.set_yticks(range(len(active)), labels)
    ax.set(xlabel='Target', ylabel='Source',
           title='Figure 6. Directional baseline-relative prediction skill')
    fig.colorbar(im, ax=ax, label='Prediction skill')
    fig.tight_layout()
    fig.savefig(OUT / 'figure_6_pair_heatmap.png', dpi=200)
    plt.show()

## 7. Alternative results presentation — compact Table 2 and Figure 8
**Original Table 2 (preserved):** `table_2_prediction_results.csv` is the full machine-learning results table. It contains donor counts, gene counts, the selected Ridge penalty, prediction errors, baseline-relative skill, median gene-wise held-out R², and bootstrap intervals. The original table and Figures 3–6 are **not** modified.

**Table 2 (compact alternative):** `table_2_compact_prediction_results.csv` contains a report-friendly subset of up to six prespecified brain-region directions, including paired/test donor counts, median gene-wise R², its 95% bootstrap interval, and baseline-relative skill. `table_2_compact_prediction_results.png` is an optional **standalone visual table**, suitable for use separately from Figure 8. The CSV retains numeric precision; the PNG shows values rounded to three decimals.

**Figure 8 — combined alternative:** `figure_8_combined_table_r2.png` is a separate, publication-style two-panel graphic **without top titles**. **(A)** shows the concise prediction-results table (direction, paired/test donors, median R², and skill). **(B)** shows median gene-wise held-out R² for the same eligible prespecified directional region pairs (points), with 95% donor-bootstrap intervals from the existing `r2_ci_low` and `r2_ci_high` columns (horizontal lines). The caption appears **below** both panels. If `RUN_ALL_PAIRS=True`, Figure 8 still displays up to six prespecified directions for readability; the full set remains in the original Table 2 and optional Figure 6.

**Interpretation:** The interval endpoints and point estimates are taken **directly from the current Colab `results` DataFrame**; none are manually typed or inferred from screenshots. Bootstrap intervals quantify held-out-donor resampling uncertainty conditional on the fitted models, not uncertainty from re-training. Different region pairs may involve different donors and selected target genes, so cross-pair ranking remains exploratory. There is **no external disease dataset or disease-related analysis**.


In [ ]:
# Section 7: supplementary, reproducible results presentation.
# The original Table 2 and Figures 3–6 stay unchanged.
# This section creates a compact alternative table and an additional Figure 8.
import textwrap
from matplotlib.ticker import MaxNLocator

# A consistent name for each region in labels, without renaming source files.
REGION_DISPLAY_NAMES = {
    'caudate_basal_ganglia': 'Caudate',
    'putamen_basal_ganglia': 'Putamen',
    'cortex': 'Cortex',
    'frontal_cortex_ba9': 'Frontal cortex (BA9)',
    'cerebellum': 'Cerebellum',
    'cerebellar_hemisphere': 'Cerebellar hemisphere',
}

def display_region(region):
    return REGION_DISPLAY_NAMES.get(region, region.replace('_', ' ').title())

# Keep the same six illustrative directions in both panels, in the declared order.
# In all-pair mode, the complete results are still available in original Table 2.
fig8_requested_pairs = pd.DataFrame(
    ILLUSTRATIVE_PAIRS, columns=['source', 'target']
)
fig8_results = fig8_requested_pairs.merge(
    results, on=['source', 'target'], how='inner', validate='one_to_one', sort=False
).copy()
if fig8_results.empty:
    raise ValueError('None of the illustrative region pairs have prediction results.')

fig8_results['direction'] = [
    f'{display_region(source)} → {display_region(target)}'
    for source, target in zip(fig8_results['source'], fig8_results['target'])
]
fig8_results['paired_test'] = [
    f'{int(paired)} / {int(test)}'
    for paired, test in zip(fig8_results['n_paired'], fig8_results['n_test'])
]

# Report-friendly compact table: this is an ADDITION, not a replacement.
compact_table_2 = fig8_results[[
    'direction', 'n_paired', 'n_test', 'median_gene_r2',
    'r2_ci_low', 'r2_ci_high', 'skill'
]].rename(columns={
    'direction': 'prediction_direction',
    'n_paired': 'paired_donors',
    'n_test': 'test_donors',
    'median_gene_r2': 'median_test_gene_r2',
    'r2_ci_low': 'r2_95ci_lower',
    'r2_ci_high': 'r2_95ci_upper',
    'skill': 'baseline_relative_skill'
})
compact_table_2.to_csv(OUT / 'table_2_compact_prediction_results.csv', index=False)

# Format only the human-readable display; underlying CSV precision is unchanged.
rendered_rows = [
    [row.direction.replace(' → ', ' to '), row.paired_test,
     f'{row.median_gene_r2:.3f}', f'{row.skill:.3f}']
    for row in fig8_results.itertuples(index=False)
]
table_headers = ['Prediction direction', 'Paired / test', 'Median R²', 'Skill']

def draw_compact_table(ax):
    ax.axis('off')
    table = ax.table(
        cellText=rendered_rows,
        colLabels=table_headers,
        cellLoc='left', colLoc='left',
        colWidths=[0.56, 0.16, 0.14, 0.14],
        bbox=[0.0, 0.0, 1.0, 1.0]
    )
    table.auto_set_font_size(False)
    table.set_fontsize(10.5 if len(rendered_rows) <= 6 else 9.0)
    for (row_idx, col_idx), cell in table.get_celld().items():
        cell.set_edgecolor('#dce3e9')
        cell.set_linewidth(0.4)
        cell.PAD = 0.045
        if row_idx == 0:
            cell.set_facecolor('#dfe8ee')
            cell.set_text_props(fontweight='bold', color='#183345')
        else:
            cell.set_facecolor('#f5f8fa' if row_idx % 2 == 0 else 'white')
            cell.set_text_props(color='#22313c')
    return table

# Standalone compact alternative to Table 2.
standalone_height = max(2.6, 0.47 * (len(fig8_results) + 1))
fig_table, ax_table_only = plt.subplots(figsize=(12.6, standalone_height))
draw_compact_table(ax_table_only)
fig_table.savefig(OUT / 'table_2_compact_prediction_results.png', dpi=300,
                  bbox_inches='tight', facecolor='white')
plt.show()

# Two-panel composite Figure 8: no top titles, panel letters, bottom caption.
fig8_height = max(8.8, 7.0 + 0.25 * len(fig8_results))
fig8 = plt.figure(figsize=(13.6, fig8_height), facecolor='white')
ax_a = fig8.add_axes([0.05, 0.565, 0.90, 0.30])
draw_compact_table(ax_a)
fig8.text(0.05, 0.885, 'A', ha='left', va='bottom', fontsize=17, fontweight='bold')

ax_b = fig8.add_axes([0.37, 0.21, 0.57, 0.29])
y = np.arange(len(fig8_results))
points = fig8_results['median_gene_r2'].to_numpy(dtype=float)
ci_lower = fig8_results['r2_ci_low'].to_numpy(dtype=float)
ci_upper = fig8_results['r2_ci_high'].to_numpy(dtype=float)

# Draw intervals from the ACTUAL donor-bootstrap values. Missing intervals are
# left blank rather than replaced by guessed or approximated limits.
for pos, low, high in zip(y, ci_lower, ci_upper):
    if np.isfinite(low) and np.isfinite(high):
        ax_b.plot([low, high], [pos, pos], color='#357da8', linewidth=2.6,
                  solid_capstyle='round', zorder=2)
valid_points = np.isfinite(points)
ax_b.scatter(points[valid_points], y[valid_points], s=62,
             color='#163b54', zorder=3)
ax_b.set_yticks(y)
ax_b.set_yticklabels(fig8_results['direction'].tolist(), fontsize=10)
ax_b.invert_yaxis()
ax_b.axvline(0, linestyle='--', color='#777777', linewidth=1.2, alpha=0.8)
ax_b.set_xlabel('Median gene-wise test R²', fontsize=11)
ax_b.grid(axis='x', alpha=0.23)
ax_b.xaxis.set_major_locator(MaxNLocator(nbins=7))
ax_b.set_axisbelow(True)
ax_b.spines[['top', 'right']].set_visible(False)

# Include zero and every valid interval end without assuming R² is positive.
limits = np.concatenate([points, ci_lower, ci_upper, [0.0]])
limits = limits[np.isfinite(limits)]
if len(limits):
    lowest, highest = float(limits.min()), float(limits.max())
    margin = max(0.04, 0.07 * (highest - lowest))
    ax_b.set_xlim(lowest - margin, highest + margin)
fig8.text(0.05, 0.50, 'B', ha='left', va='bottom', fontsize=17, fontweight='bold')

# Caption at the bottom, rather than headings above either panel.
caption = (
    f'Fig. 8. (A) Summary of {len(fig8_results)} directional GTEx brain-region comparisons: '
    'paired and held-out test donor counts, median gene-wise test R², and '
    'prediction skill relative to the development-set mean-expression baseline. '
    '(B) Median gene-wise test R² (points) with 95% held-out-donor bootstrap '
    'intervals (horizontal lines). Intervals are conditional on fitted models; '
    'the available donors and selected target genes may differ between pairs.'
)
fig8.text(0.05, 0.06, textwrap.fill(caption, width=145),
          fontsize=10.2, ha='left', va='bottom', color='#243443')
fig8.savefig(OUT / 'figure_8_combined_table_r2.png', dpi=300,
             bbox_inches='tight', facecolor='white')
plt.show()

print('Added compact alternative: table_2_compact_prediction_results.csv')
print('Added standalone table:  table_2_compact_prediction_results.png')
print('Added combined Fig. 8:   figure_8_combined_table_r2.png')
print('All three files are saved in', OUT)


## 8. Reporting results and reproducibility
Use `table_1_dataset_summary.csv`, `donor_overlap.csv` and Figures 1–2 to describe which regions and donors were available. Use `table_2_prediction_results.csv` and Figures 3 and 5 to answer the research question: **How accurately does expression predict another region in held-out donors, and how does accuracy vary across source → target directions?** `table_3_gene_prediction_results.csv` provides per-gene test R², MAE and RMSE for auditing the reported summaries. Figure 4 illustrates one gene's predicted versus observed expression, and Figure 6 displays the full directional matrix if all-pair mode is enabled. For a cleaner presentation of the six default directions, use the alternative `table_2_compact_prediction_results.csv` / `.png`, or the combined **Figure 8**, where panel A is a concise table and panel B is the median gene-wise R² plot with bootstrap intervals.

Report both median gene-wise R² and baseline-relative prediction skill, noting that the two metrics use different reference means. Include negative scores rather than concealing them. Comparisons between pairs are exploratory because the available donor set and selected target genes differ, and pairs can reuse donors. The bootstrap intervals describe variability from held-out donor resampling, not complete model-development uncertainty.

Before making strong claims from pair rankings, consider repeated donor splits and sensitivity analyses on common donors/common target genes, and assess the effect of normalization or available confounder metadata. Record GTEx V11 release provenance, file hashes, software versions, and data-use conditions. The ZIP export excludes files marked `PRIVATE` containing donor IDs; verify data-sharing permissions before posting any additional outputs.

For BMEN 600, caption figures, document preprocessing, identify the donor as the independent case, and describe leakage prevention, model selection, the baseline, uncertainty and limitations. Cite the GitHub commit and provide truthful author contributions and an AI-assistance statement.

**Additional files:** `table_2_compact_prediction_results.csv`, `table_2_compact_prediction_results.png`, and `figure_8_combined_table_r2.png` are generated in Section 7, saved to `OUT`, and included in the public results ZIP. They are alternate visual presentations, not independent analyses.


In [ ]:
metadata = {
    'research_question': (
        'How accurately can gene expression in one human brain region predict expression '
        'in another in unseen GTEx donors, and how does predictive performance vary across region pairs?'
    ),
    'seed': SEED,
    'run_all_pairs': RUN_ALL_PAIRS,
    'include_spinal_cord': INCLUDE_SPINAL_CORD,
    'normalization': 'log2(CPM+1), using per-sample total gene counts',
    'source_genes': N_SOURCE_GENES,
    'target_genes': N_TARGET_GENES,
    'alphas': ALPHAS,
    'bootstraps': N_BOOTSTRAPS,
    'python': platform.python_version(),
    'numpy': np.__version__,
    'pandas': pd.__version__,
    'sklearn': sklearn.__version__,
    'input_files': records,
    'illustrative_pairs': ILLUSTRATIVE_PAIRS,
    'github_release': DATA_URL
}
(OUT / 'analysis_metadata.json').write_text(json.dumps(metadata, indent=2))

# Package only files that are safe for a public results ZIP.
PUBLIC = Path('/content/gtex_public_results')
PUBLIC.mkdir(exist_ok=True)
for old in PUBLIC.iterdir():
    if old.is_file():
        old.unlink()
for p in OUT.iterdir():
    if p.is_file() and 'PRIVATE' not in p.name:
        shutil.copy2(p, PUBLIC / p.name)
archive = shutil.make_archive('/content/gtex_results_public', 'zip', PUBLIC)
from google.colab import files
files.download(archive)

## 9. References (IEEE style)
[1] GTEx Consortium, “The GTEx Consortium atlas of genetic regulatory effects across human tissues,” *Science*, vol. 369, pp. 1318–1330, 2020. doi: [10.1126/science.aaz1776](https://doi.org/10.1126/science.aaz1776). This provides GTEx background but describes V8, **not** the supplied V11 release. Record verified V11 source/release details separately.

[2] M. D. Robinson and A. Oshlack, “A scaling normalization method for differential expression analysis of RNA-seq data,” *Genome Biology*, vol. 11, R25, 2010. doi: [10.1186/gb-2010-11-3-r25](https://doi.org/10.1186/gb-2010-11-3-r25). Used to discuss compositional bias; TMM is **not** implemented.

[3] A. E. Hoerl and R. W. Kennard, “Ridge regression: Biased estimation for nonorthogonal problems,” *Technometrics*, vol. 12, no. 1, pp. 55–67, 1970. doi: [10.1080/00401706.1970.10488634](https://doi.org/10.1080/00401706.1970.10488634).

[4] S. Varma and R. Simon, “Bias in error estimation when using cross-validation for model selection,” *BMC Bioinformatics*, vol. 7, art. 91, 2006. doi: [10.1186/1471-2105-7-91](https://doi.org/10.1186/1471-2105-7-91). This notebook uses separate development-validation and test splits, not nested cross-validation.

Software: [scikit-learn Ridge](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Ridge.html), [StandardScaler](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.StandardScaler.html).